# 🧬 Bioinformatics Trends Analysis (2016–2026)
## Reproducible Notebook for LinkedIn Content

---

## 1. Introduction and Research Plan

### Objective
This notesssbook analyzes the evolution of bioinformatics over the last decade (2016–2026), quantifying the growth of disciplinary sub-areas through bibliometric analysis, market data, and tracking genomic sequencing costs.

### Research Methodology
1. **Bibliometric analysis**: Query to PubMed E-utilities API for annual publication count per search term (2016–2025)
2. **Market data**: Collection from industry reports (Reports and Data, SNS Insider, Nova One Advisor, Coherent Market Insights, Precedence Research)
3. **Sequencing costs**: NHGRI data (genome.gov) - official historical series
4. **Milestone timeline**: Synthesis from primary and secondary literature

### PubMed Search Strategy
- Used API: `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi`
- Parameters: `datetype=pdat` (publication date), annual windows 01/01–31/12
- Rate limiting: 0.35 secondi tra le richieste (NCBI limit: ~3 req/s without API key)
- Cache: results saved in CSV to avoid re-querying

### Main Sources
| Source | URL | Type |
|-------|-----|------|
| PubMed E-utilities | https://www.ncbi.nlm.nih.gov/books/NBK25499/ | API |
| NHGRI Sequencing Cost | https://www.genome.gov/about-genomics/fact-sheets/DNA-Sequencing-Costs-Data | Official data |
| StartUs Insights | https://www.startus-insights.com/innovators-guide/bioinformatics-trends/ | Report |
| CSET Georgetown | https://cset.georgetown.edu/publication/ai-and-the-future-of-life-sciences/ | Report |
| PharmiWeb/SNS Insider | https://www.pharmiweb.com/press-release/2024-07-16/bioinformatics-market-size-worth-usd-13614-million-in-2023 | Market data |
| Coherent Market Insights | https://www.coherentmarketinsights.com/market-insight/bioinformatics-market-2177 | Projections |
| BioSpace/Precedence | https://www.biospace.com/bioinformatics-market | Market data |
| Nova One Advisor | https://www.novaoneadvisor.com/report/bioinformatics-market | Market data |
| Nature Methods | https://www.nature.com/articles/s41592-020-01042-x | Journal |
| Mordor Intelligence | https://www.mordorintelligence.com/industry-reports/bioinformatics-market | Report |
| IntechOpen Bioinformatics | https://www.intechopen.com/books/1000107 | Review |
| ZAGENO Blog | https://zageno.com/l/bioinformatics-trends | Trends |
| Fios Genomics | https://fiosgenomics.com/bioinformatics-trends/ | Trends |


## 2. Setup and Dependencies

In [ ]:
# Install dependencies (uncomment if necessary)
# !pip install matplotlib seaborn pandas numpy requests nbformat

import os
import time

import warnings
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Professional style for LinkedIn ──────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.color': '#CCCCCC',
    'grid.linewidth': 0.6,
    'grid.alpha': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'legend.framealpha': 0.9,
    'lines.linewidth': 2.5,
    'lines.markersize': 7,
})

# ── Output directory ──────────────────────────────────────────────────────────
CHARTS_DIR = '/Users/marcoanteghini/Research/bioinformatics_job_market'
os.makedirs(CHARTS_DIR, exist_ok=True)
CACHE_CSV  = '/Users/marcoanteghini/Research/bioinformatics_job_market/pubmed_cache.csv'

# ── Colorblind-friendly palette (Okabe-Ito + extended) ─────────────────────────
PALETTE = [
    '#0173B2', '#DE8F05', '#029E73', '#D55E00', '#CC78BC',
    '#CA9161', '#FBAFE4', '#949494', '#ECE133', '#56B4E9',
    '#009E73', '#F0E442',
]

YEARS = list(range(2016, 2026))  # 2016–2025

print("✅ Setup completed")
print(f"   matplotlib {matplotlib.__version__} | pandas {pd.__version__} | numpy {np.__version__}")


## 3. PubMed Data Collection

We use the NCBI E-utilities API to retrieve the number of publications indexed in PubMed for each search term and for each year from 2016 to 2025.

**Documentazione API**: https://www.ncbi.nlm.nih.gov/books/NBK25499/


In [ ]:
import requests
import xml.etree.ElementTree as ET
import time

def get_pubmed_count(term, year):
    """Query PubMed for the publication count for a given term and year."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "rettype": "count",
        "mindate": f"{year}/01/01",
        "maxdate": f"{year}/12/31",
        "datetype": "pdat"
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        root = ET.fromstring(response.content)
        count = int(root.find("Count").text)
        time.sleep(0.35)  # Rate limiting NCBI
        return count
    except Exception as e:
        print(f"  ⚠️  Error for '{term}' ({year}): {e}")
        return None


# ── Search terms (21 categories) ────────────────────────────────────────
SEARCH_TERMS = {

    # ── Basic Bioinformatics ────────────────────────────────────────────────
    'Bioinformatics (general)': 'bioinformatics',

    # ── Genomics & Sequencing ─────────────────────────────────────────────
    'Single-Cell Sequencing':   '"single cell sequencing" OR "single-cell RNA-seq" OR "scRNA-seq"',
    'Spatial Transcriptomics':  '"spatial transcriptomics"',
    'Long-Read Sequencing':     '"long read sequencing" OR "nanopore sequencing" OR "PacBio sequencing"',
    'Metagenomics & Microbiome':'metagenomics OR "microbiome analysis"',
    # Buenrostro 2015 → grande boom 2020+
    'Epigenomica & scATAC-seq': 'scATAC-seq OR "single-cell ATAC" OR ("chromatin accessibility" AND "single cell")',

    # ── Editing & Precision Medicine ───────────────────────────────────
    'CRISPR & Bioinformatica':           'CRISPR AND bioinformatics',
    'Precision Medicine & Genomics': '"precision medicine" AND genomics',

    # ── Proteomics & 3D Structure ─────────────────────────────────────────────
    'Proteomica':                 'proteomics AND bioinformatics',
    # AlphaFold2 (DeepMind 2021), ESMFold, RoseTTAFold
    'AlphaFold & Struttura Prot.':'AlphaFold OR "protein structure prediction"',
    # cryoDRGN, ModelAngelo, DeepEMhancer
    'Cryo-EM & Bioinformatica':   '("cryo-EM" OR "cryo-electron microscopy") AND (bioinformatics OR "structure determination" OR "structure prediction")',

    # ── AI & Machine Learning ─────────────────────────────────────────────────
    'Machine Learning & Bio':  '"machine learning" AND bioinformatics',
    'Deep Learning & Bio':     '"deep learning" AND bioinformatics',
    'Multi-Omics':             '"multi-omics" OR multiomics',
    # DGL-LifeSci, Graphein, MolGNN — boom drug discovery 2021+
    'Graph Neural Networks in Bio': '"graph neural network" AND (genomics OR proteomics OR "drug discovery" OR bioinformatics)',

    # ── Protein Language Models (pLMs) ────────────────────────────────────────
    # ESM-1b / ESM-2 / ESMFold (Meta, Lin et al. Science 2023)
    # ProtBERT / ProtT5 / ProtXLNet (Elnaggar et al. 2021 — ProtTrans)
    # UniRep (Alley et al. 2019) | ProteinBERT (Brandes et al. 2022)
    # ProGen / ProGen2 (Salesforce) | Ankh (2023) | SaProt (2023)
    'Protein Language Models': (
        '"protein language model" OR "pLM"[tiab] OR ESMFold OR "ESM-2"[tiab] '
        'OR ProtTrans OR ProtBERT OR ProtT5[tiab] OR UniRep[tiab] '
        'OR ProteinBERT OR ProGen[tiab] OR Ankh[tiab] OR SaProt[tiab]'
    ),

    # ── DNA / RNA Foundation Models ───────────────────────────────────────────
    # DNABERT / DNABERT-2 (Ji et al.) | Nucleotide Transformer (InstaDeep/EMBL 2023)
    # HyenaDNA (Hazy Research 2023) | Caduceus (2024) | Evo (Arc Institute 2024)
    # GPN (Benegas et al.) | RNA-FM | RNA-MSM
    'DNA/RNA Foundation Models': (
        '"DNA language model" OR "genomic language model" OR "nucleotide transformer" '
        'OR DNABERT[tiab] OR HyenaDNA[tiab] OR Caduceus[tiab] OR "RNA-FM"[tiab]'
    ),

    # ── Generative AI & Protein Design ───────────────────────────────────────
    # RFDiffusion (Baker Lab 2023), ProteinMPNN, Chroma, FrameDiff, Genie, Framediff
    'Generative AI & Protein Design': (
        '("de novo protein design" OR "protein design") AND '
        '("diffusion model" OR "generative model" OR "deep learning")'
    ),

    # ── Cell Foundation Models ────────────────────────────────────────────────
    # scGPT (Wang et al. 2023) | Geneformer (Theodoris et al. Nature 2023)
    # scBERT (Yang et al. 2022) | CellPLM | Universal Cell Embeddings (UCE)
    'Cell Foundation Models': (
        'scGPT[tiab] OR Geneformer[tiab] OR scBERT[tiab] OR CellPLM[tiab] '
        'OR "single cell foundation model" OR "cell language model"'
    ),
}

print(f"📊 Terms to query: {len(SEARCH_TERMS)}")
print(f"📅 Years: {YEARS[0]}–{YEARS[-1]}")
print(f"📁 Cache CSV: {CACHE_CSV}")


In [ ]:
# ── Cache loading or PubMed query ─────────────────────────────────────────
import os

if os.path.exists(CACHE_CSV) and os.path.getsize(CACHE_CSV) > 0:
    print(f"✅ Cache found: {CACHE_CSV}  →  loading saved data")
    df_pubmed = pd.read_csv(CACHE_CSV, index_col=0)
    df_pubmed.columns = df_pubmed.columns.astype(int)
    print(f"   Shape: {df_pubmed.shape}  |  anni: {list(df_pubmed.columns)}")
else:
    if os.path.exists(CACHE_CSV):
        print(f"⚠️  Cache found ma vuota: {CACHE_CSV}  →  re-running PubMed query")
    else:
        print("🔍 Cache not found → PubMed API query in progress...")
    print(f"   (estimated ~{len(SEARCH_TERMS) * len(YEARS) * 0.35 / 60:.1f} minutes)")
    
    results = {}
    total = len(SEARCH_TERMS) * len(YEARS)
    done = 0
    for label, term in SEARCH_TERMS.items():
        counts = {}
        for year in YEARS:
            count = get_pubmed_count(term, year)
            counts[year] = count if count is not None else np.nan
            done += 1
            if done % 10 == 0:
                print(f"   {done}/{total} queries completed …")
        results[label] = counts
        print(f"  ✓ {label}")
    
    df_pubmed = pd.DataFrame(results).T
    df_pubmed.to_csv(CACHE_CSV)
    print(f"\n✅ Data saved in cache: {CACHE_CSV}")

df_pubmed


In [ ]:
# ── Quick descriptive statistics ───────────────────────────────────────────
print("📊 Total counts per term (sum 2016-2025):")
totals = df_pubmed.sum(axis=1).sort_values(ascending=False)
for label, val in totals.items():
    print(f"   {label:40s}: {val:,.0f}")

print("\n📈 Growth % (2016→2025):")
growth = ((df_pubmed[2025] - df_pubmed[2016]) / df_pubmed[2016] * 100).sort_values(ascending=False)
for label, val in growth.items():
    print(f"   {label:40s}: {val:+.1f}%")


## 4. Market Data and Sequencing Costs

Data collected manually from market reports and institutional sources. Citations included in the code.


In [ ]:
# ── a) Bioinformatics market size (billion USD) ──────────────────────
# Sources: Reports and Data, SNS Insider/PharmiWeb, BioSpace/Precedence Research,
#         Nova One Advisor, Coherent Market Insights
# URL: https://www.pharmiweb.com/press-release/2024-07-16/bioinformatics-market-size-worth-usd-13614-million-in-2023
# URL: https://www.novaoneadvisor.com/report/bioinformatics-market
# URL: https://www.coherentmarketinsights.com/market-insight/bioinformatics-market-2177
# URL: https://www.biospace.com/bioinformatics-market

market_data = {
    'year': [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026],
    'market_bn_usd': [5.5, 6.0, 6.41, 7.2, 8.1, 9.2, 10.5, 13.6, 14.89, 17.83, 39.22],
    'notesss': [
        'Estimate',
        'Estimate',
        'Reports and Data',
        'Estimate',
        'Estimate',
        'Estimate',
        'Estimate',
        'SNS Insider / PharmiWeb',
        'Average BioSpace & Precedence Research (13.12–16.66)',
        'Nova One Advisor',
        'Projection Coherent Market Insights',
    ]
}
df_market = pd.DataFrame(market_data)
print("✅ Dati mercato caricati:")
print(df_market.to_string(index=False))


In [ ]:
# ── b) Sequencing cost per genome (USD) ─────────────────────────────────
# Source: NHGRI – National Human Genome Research Institute
# URL: https://www.genome.gov/about-genomics/fact-sheets/DNA-Sequencing-Costs-Data

sequencing_data = {
    'year': [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014,
             2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'costo_usd': [14_000_000, 10_000_000, 3_000_000, 100_000, 50_000, 10_000,
                  7_000, 6_000, 5_000, 4_000, 1_500, 1_200, 1_100, 1_000,
                  800, 600, 500, 400, 200, 150],
    'milestone': [
        '', '', 'Rivoluzione NGS', '', '', '', '', '', 'Genome <$5k',
        '', '', '', '', '', '', '', '', '', 'Illumina ~$200', '~$100–200'
    ]
}
df_seq = pd.DataFrame(sequencing_data)
print("✅ Dati costo sequenziamento caricati:")
print(df_seq.to_string(index=False))


## 5. Visualizations

All charts are saved as high-resolution PNGs in `/home/user/workspace/charts/`.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 1 – PubMed Publications by Bioinformatics Area (2016–2025)
# ════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 8))

growth_pct = ((df_pubmed[2025] - df_pubmed[2016]) / df_pubmed[2016] * 100)
top3 = set(growth_pct.nlargest(3).index)

for i, (label, row) in enumerate(df_pubmed.iterrows()):
    color = PALETTE[i % len(PALETTE)]
    lw = 3.5 if label in top3 else 2.0
    alpha = 1.0 if label in top3 else 0.75
    zorder = 5 if label in top3 else 2
    marker = 'o' if label in top3 else 's'
    ax.plot(YEARS, row.values, color=color, linewidth=lw,
            alpha=alpha, zorder=zorder, marker=marker,
            markersize=7 if label in top3 else 5, label=label)

ax.set_title('PubMed Publications by Bioinformatics Area (2016–2025)',
             fontsize=17, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Number of Publications', fontsize=13)
ax.set_xticks(YEARS)
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Yeartazione top 3
for label in top3:
    last_val = df_pubmed.loc[label, 2025]
    ax.annotate(f'⭐ {label}\n+{growth_pct[label]:.0f}%',
                xy=(2025, last_val),
                xytext=(10, 0), textcoords='offset points',
                fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

ax.legend(loc='upper left', bbox_to_anchor=(0, 1), ncol=1,
          fontsize=10, framealpha=0.95)
fig.text(0.99, 0.01, 'Source: PubMed E-utilities API (NCBI) | github analisi bioinformatica 2026',
         ha='right', va='bottom', fontsize=8, color='gray', style='italic')
plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart1_pubmed_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 1 saved")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 1 – PubMed Publications by Bioinformatics Area (2016–2025)
#           (excl. "Bioinformatics (general)" – different scale)
# ════════════════════════════════════════════════════════════════════════════

# ── Label translation map (CSV row → English display) ───────────────────────
LABEL_EN = {
    'Bioinformatics (general)':         'Bioinformatics (general)',
    'Single-Cell Sequencing':            'Single-Cell Sequencing',
    'Spatial Transcriptomics':           'Spatial Transcriptomics',
    'Long-Read Sequencing':              'Long-Read Sequencing',
    'Metagenomics & Microbiome':         'Metagenomics & Microbiome',
    'Epigenomica & scATAC-seq':          'Epigenomics & scATAC-seq',
    'CRISPR & Bioinformatica':           'CRISPR & Bioinformatics',
    'Precision Medicine & Genomics': 'Precision Medicine & Genomics',
    'Proteomica':                        'Proteomics',
    'AlphaFold & Struttura Prot.':       'AlphaFold & Protein Structure',
    'Cryo-EM & Bioinformatica':          'Cryo-EM & Bioinformatics',
    'Machine Learning & Bio':            'Machine Learning & Bio',
    'Deep Learning & Bio':               'Deep Learning & Bio',
    'Multi-Omics':                       'Multi-Omics',
    'Graph Neural Networks in Bio':      'Graph Neural Networks in Bio',
    'Protein Language Models':           'Protein Language Models',
    'DNA/RNA Foundation Models':         'DNA/RNA Foundation Models',
    'Generative AI & Protein Design':    'Generative AI & Protein Design',
    'Cell Foundation Models':            'Cell Foundation Models',
}

# ── Exclude from plot ────────────────────────────────────────────────────────
EXCLUDE_FROM_CHART1 = {'Bioinformatics (general)'}
df_plot = df_pubmed.drop(index=[i for i in EXCLUDE_FROM_CHART1 if i in df_pubmed.index])

# ── Lines to annotate (specific selection) ───────────────────────────────────
highest_label = df_plot[2025].idxmax()
ANNOTATE_LABELS = {
    highest_label,                        # line with most publications in 2025
    'Protein Language Models',
    'Deep Learning & Bio',
    'Precision Medicine & Genomics',
}
ANNOTATE_LABELS = {l for l in ANNOTATE_LABELS if l in df_plot.index}

# ── Build a colour map (consistent index → colour) ───────────────────────────
label_list = list(df_plot.index)
color_map = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(label_list)}

growth_pct = ((df_plot[2025] - df_plot[2016]) / df_plot[2016] * 100)

fig, ax = plt.subplots(figsize=(15, 8))

for label, row in df_plot.iterrows():
    is_ann  = label in ANNOTATE_LABELS
    color   = color_map[label]
    lw      = 3.0 if is_ann else 1.6
    alpha   = 1.0 if is_ann else 0.55
    zorder  = 5 if is_ann else 2
    marker  = 'o' if is_ann else 's'
    ms      = 7   if is_ann else 3
    ax.plot(YEARS, row.values, color=color, linewidth=lw,
            alpha=alpha, zorder=zorder, marker=marker,
            markersize=ms, label=LABEL_EN.get(label, label))

# ── Yeartations – sorted by 2025 value, staggered y-offsets ─────────────────
ann_sorted = sorted(ANNOTATE_LABELS, key=lambda l: df_plot.loc[l, 2025], reverse=True)

# Manual y-offsets (points) per rank to avoid overlap
y_offsets = {
    ann_sorted[0]: +30,   # highest  → push up
    ann_sorted[1]: +15,   # second
    ann_sorted[2]: -25,   # third    → push down
    ann_sorted[3]: -40,   # lowest   → push further down
}

for label in ann_sorted:
    val   = df_plot.loc[label, 2025]
    g     = growth_pct[label]
    color = color_map[label]
    en    = LABEL_EN.get(label, label)
    dy    = y_offsets.get(label, 0)

    ax.annotate(
        f'{en}\n+{g:.0f}%',
        xy=(2025, val),
        xytext=(20, dy), textcoords='offset points',
        fontsize=8.5, fontweight='bold', color=color, va='center',
        arrowprops=dict(arrowstyle='->', color=color, lw=1.1),
        bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=color, alpha=0.85),
    )

ax.set_title('PubMed Publications by Bioinformatics Area (2016–2025)',
             fontsize=17, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Number of Publications', fontsize=13)
ax.set_xticks(YEARS)
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

ax.legend(loc='upper left', bbox_to_anchor=(0, 1), ncol=1,
          fontsize=9, framealpha=0.95)
fig.text(0.99, 0.01,
         'Source: PubMed E-utilities API (NCBI) | Bioinformatics Trends Analysis 2026',
         ha='right', va='bottom', fontsize=8, color='gray', style='italic')

plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart1_pubmed_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 1 saved")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 2 – Growth Relativa delle Aree (Index 2016 = 100)
# ════════════════════════════════════════════════════════════════════════════

# Normalizzazione: 2016 = 100
base_2016 = df_pubmed[2016].replace(0, np.nan)
df_indexed = df_pubmed.astype(float).div(base_2016, axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 8))

for i, (label, row) in enumerate(df_indexed.iterrows()):
    color = PALETTE[i % len(PALETTE)]
    lw = 3.0 if label in top3 else 1.8
    alpha = 1.0 if label in top3 else 0.7
    ax.plot(YEARS, row.values, color=color, linewidth=lw,
            alpha=alpha, marker='o', markersize=6, label=label)

ax.axhline(100, color='black', linewidth=1.5, linestyle='--', alpha=0.5,
           label='Baseline 2016 = 100')

ax.set_title('Growth Relativa delle Aree Bioinformatiche (Index 2016 = 100)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Index (2016 = 100)', fontsize=13)
ax.set_xticks(YEARS)
ax.legend(loc='upper left', bbox_to_anchor=(0, 1), ncol=1, fontsize=9.5, framealpha=0.95)
fig.text(0.99, 0.01, 'Source: PubMed E-utilities API (NCBI)',
         ha='right', va='bottom', fontsize=8, color='gray', style='italic')
plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart2_relative_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 2 saved")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 3 – Growth Ranking: Raw % vs CAGR  (2016 → 2025)
# ════════════════════════════════════════════════════════════════════════════
#
# Two metrics are shown side-by-side to expose a well-known visualisation
# pitfall when measuring growth from count data:
#
#   PANEL A — Raw percentage growth  (%Δ = (2025 – 2016) / 2016 × 100)
#   ─────────────────────────────────────────────────────────────────────
#   Intuitive and easy to communicate, but STRONGLY biased toward sub-areas
#   that started from a near-zero baseline.  Example: Spatial Transcriptomics
#   had just 2 PubMed entries in 2016 and reached 2,458 in 2025 → +122,800%.
#   This dwarfs Multi-Omics (+4,083%) even though the latter represents far
#   more absolute scientific output.  A minimum threshold of ≥10 publications
#   in the baseline year is applied to exclude the most extreme outliers.
#
#   PANEL B — Compound Annual Growth Rate  (CAGR = (2025/2016)^(1/9) – 1)
#   ─────────────────────────────────────────────────────────────────────
#   Normalises the 9-year window into a per-year rate, compressing extreme
#   outliers and giving a fairer picture of sustained momentum.  A field
#   growing at 40% CAGR doubles every ~2 years; at 20% it doubles every ~4.
#   CAGR is standard in market-research and finance reports; it is robust to
#   baseline effects because the exponent (1/n) penalises short bursts.
#
#   Take-away: fields that rank high in BOTH panels are genuine long-run
#   trends; fields that rank high only in Panel A likely started from a
#   very small count and should be interpreted with caution.
# ════════════════════════════════════════════════════════════════════════════

MIN_PUBS_2016 = 10        # minimum publications in 2016 to be included
N_YEARS       = 9         # 2025 − 2016

# ── Shared filtered dataframe ─────────────────────────────────────────────────
df_g3 = df_pubmed.drop(index=[i for i in EXCLUDE_FROM_CHART1 if i in df_pubmed.index])
df_g3 = df_g3[df_g3[2016] >= MIN_PUBS_2016].copy()

# ── Metric A: raw growth % ────────────────────────────────────────────────────
raw_growth = ((df_g3[2025] - df_g3[2016]) / df_g3[2016] * 100)
raw_growth = raw_growth.replace([np.inf, -np.inf], np.nan).dropna()
raw_sorted = raw_growth.sort_values(ascending=True)
raw_sorted.index = [LABEL_EN.get(l, l) for l in raw_sorted.index]

# ── Metric B: CAGR ────────────────────────────────────────────────────────────
cagr = ((df_g3[2025] / df_g3[2016]) ** (1 / N_YEARS) - 1) * 100
cagr = cagr.replace([np.inf, -np.inf], np.nan).dropna()
cagr_sorted = cagr.sort_values(ascending=True)
cagr_sorted.index = [LABEL_EN.get(l, l) for l in cagr_sorted.index]

# ── Consistent colour map (label → colour, same across both panels) ───────────
color_map_g3 = {label: PALETTE[i % len(PALETTE)]
                for i, label in enumerate(raw_sorted.index)}

# ── Drawing helper ────────────────────────────────────────────────────────────
def _barh_panel(ax, series, xlabel, title, fmt_fn):
    colors = [color_map_g3.get(l, '#888888') for l in series.index]
    bars = ax.barh(range(len(series)), series.values,
                   color=colors, edgecolor='white', linewidth=1.1, height=0.72)
    ax.set_yticks(range(len(series)))
    ax.set_yticklabels(series.index, fontsize=10.5)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_fn))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.35)

    max_val = float(series.values.max())
    for bar, val, label in zip(bars, series.values, series.index):
        ax.text(val + max_val * 0.015,
                bar.get_y() + bar.get_height() / 2,
                fmt_fn(val, None),
                va='center', ha='left', fontsize=9.5, fontweight='bold',
                color=color_map_g3.get(label, 'black'))
    ax.set_xlim(0, max_val * 1.25)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Bioinformatics Sub-Area Growth Ranking  (2016 → 2025)',
             fontsize=18, fontweight='bold', y=1.02)

_barh_panel(
    ax1, raw_sorted,
    xlabel='Total Growth %  (2016 → 2025)',
    title='Panel A  –  Raw Growth %\n(baseline ≥ 10 publications in 2016)',
    fmt_fn=lambda x, _: f'+{x:,.0f}%',
)

_barh_panel(
    ax2, cagr_sorted,
    xlabel='CAGR  (% per year)',
    title='Panel B  –  Compound Annual Growth Rate (CAGR)\n(9-year window, same dataset)',
    fmt_fn=lambda x, _: f'+{x:.1f}%/yr',
)

# ── Shared caption ────────────────────────────────────────────────────────────
caption = (
    'Panel A: raw %-change is intuitive but biased toward near-zero baselines '
    '(min. 10 pubs in 2016 applied to reduce distortion).   '
    'Panel B: CAGR smooths baseline effects and reflects sustained year-over-year momentum.   '
    'Fields ranking high in both panels represent genuinely robust long-term trends.\n'
    'Source: PubMed E-utilities API (NCBI)  |  Bioinformatics Trends Analysis 2026'
)
fig.text(0.5, -0.02, caption, ha='center', va='top',
         fontsize=8.5, color='gray', style='italic')

plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart3_growth_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 3 saved")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 4 – Heatmap: Year-over-Year (YoY) % Change by Bioinformatics Area
# ════════════════════════════════════════════════════════════════════════════

# ── Prepare data ──────────────────────────────────────────────────────────────
df_heat = df_pubmed.drop(index=[i for i in EXCLUDE_FROM_CHART1 if i in df_pubmed.index])

# YoY % change; inf arises when a year has 0 publications → replace with NaN
df_yoy = df_heat.astype(float).pct_change(axis=1) * 100
df_yoy = df_yoy.iloc[:, 1:]                               # drop first column (all NaN)
df_yoy = df_yoy.replace([np.inf, -np.inf], np.nan)

# Translate row labels to English
df_yoy.index = [LABEL_EN.get(l, l) for l in df_yoy.index]

# ── Robust colour-scale bounds (symmetric ±95th-percentile of |values|) ──────
# Using a percentile cap prevents a handful of extreme outliers (e.g. a field
# jumping from 2 → 30 publications = +1,400%) from washing out the rest of
# the colormap, while the annotated numbers still show the true values.
flat  = df_yoy.values.flatten()
flat  = flat[~np.isnan(flat)]
vlim  = float(np.percentile(np.abs(flat), 95))   # symmetric ± limit

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 11))

cmap = sns.diverging_palette(10, 140, s=85, l=50, as_cmap=True)

sns.heatmap(
    df_yoy, ax=ax,
    cmap=cmap, center=0, vmin=-vlim, vmax=vlim,
    yeart=True, fmt='.0f', yeart_kws={'size': 8.5},
    linewidths=0.45, linecolor='white',
    cbar_kws={'label': 'Year-over-Year Change (%)', 'shrink': 0.75},
    mask=df_yoy.isna(),      # grey-out cells with no valid data (zero baseline)
)

ax.set_title('Year-over-Year % Change by Bioinformatics Area  (2017 → 2025)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('')
ax.set_xticklabels(df_yoy.columns, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

fig.text(
    0.99, 0.01,
    'Source: PubMed E-utilities API (NCBI)  |  '
    'Green = growth · Red = decline  |  '
    f'Grey cells = zero baseline (inf excluded)  |  '
    f'Colour scale capped at ±{vlim:.0f}% (95th percentile); labels show true values.',
    ha='right', va='bottom', fontsize=7.5, color='gray', style='italic',
)

plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart4_heatmap_yoy.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 4 saved")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 7 – Thematic Composition of Bioinformatics: 2016 vs 2025
# ════════════════════════════════════════════════════════════════════════════

# Exclude the general "bioinformatics" row and translate labels
sub_terms  = [k for k in df_pubmed.index if k not in EXCLUDE_FROM_CHART1]
data_2016  = df_pubmed.loc[sub_terms, 2016].fillna(0)
data_2025  = df_pubmed.loc[sub_terms, 2025].fillna(0)

# Translate series index to English
data_2016.index = [LABEL_EN.get(l, l) for l in data_2016.index]
data_2025.index = [LABEL_EN.get(l, l) for l in data_2025.index]

# ── Group slices < threshold into "Other" ────────────────────────────────────
def make_pie_data(series, threshold=0.03):
    total      = series.sum()
    mask_small = (series / total) < threshold
    main       = series[~mask_small]
    other_val  = series[mask_small].sum()
    if other_val > 0:
        main = pd.concat([main, pd.Series({'Other': other_val})])
    return main

d2016 = make_pie_data(data_2016)
d2025 = make_pie_data(data_2025)

# Consistent colour map across both pies (same label → same colour)
all_labels      = sorted(set(d2016.index) | set(d2025.index))
label_color_map = {lab: PALETTE[i % len(PALETTE)] for i, lab in enumerate(all_labels)}

def get_colors(labels):
    return [label_color_map.get(l, '#AAAAAA') for l in labels]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Thematic Composition of Bioinformatics: 2016 vs 2025',
             fontsize=17, fontweight='bold', y=1.01)

wedge_kw = dict(edgecolor='white', linewidth=2)

for ax_pie, data, year in [(axes[0], d2016, 2016), (axes[1], d2025, 2025)]:
    colors_pie = get_colors(data.index)
    wedges, texts, autotexts = ax_pie.pie(
        data.values,
        labels=None,
        colors=colors_pie,
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.80,
        wedgeprops=wedge_kw,
        textprops={'fontsize': 10},
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_fontweight('bold')

    ax_pie.set_title(str(year), fontsize=16, fontweight='bold')
    ax_pie.legend(
        data.index,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.30),
        ncol=1, fontsize=9, framealpha=0.9,
    )

fig.text(
    0.99, -0.03,
    'Source: PubMed E-utilities API (NCBI)  |  '
    'Slices < 3% of total are merged into "Other".',
    ha='right', va='bottom', fontsize=8, color='gray', style='italic',
)

plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart7_composition.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 7 saved")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 8 – Timeline of Bioinformatics Milestones (2016–2026)
# ════════════════════════════════════════════════════════════════════════════

milestones_timeline = [
    (2016, 'CRISPR: first clinical therapies\nDeep learning emerges in bio', '#0173B2'),
    (2017, 'Long-read sequencing\n(Nanopore, PacBio) matures', '#56B4E9'),
    (2018, 'Single-cell sequencing boom\nBERT revolutionizes NLP→bio', '#029E73'),
    (2019, 'Human genome at $1,000\nScRNA-seq Method of Year', '#009E73'),
    (2020, 'AlphaFold2 wins CASP14\nCOVID genomic surveillance\nSpatial transcriptomics MoY', '#D55E00'),
    (2021, 'AlphaFold DB: 200M+ strutture\nFirst CRISPR therapy approved (CTX001)', '#DE8F05'),
    (2022, 'Acceleration of AI/ML in bio\nChatGPT → LLM in bioinformatica', '#CC78BC'),
    (2023, 'GPT/LLM per genomica\nMulti-omics mainstream\nscGPT, ESM2', '#CA9161'),
    (2024, 'Genome at $200 (Illumina)\nAI biotech investment $5.6B\n48 precision medicine drugs FDA', '#0173B2'),
    (2025, 'Foundation models single-cell\nFederated learning in bio\nAgentic AI per drug discovery', '#029E73'),
    (2026, 'AI-therapeutics in clinical practicel practicel practice\nScalable spatial biology\nCardiovascular base editing', '#D55E00'),
]

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('white')

years_tl = [m[0] for m in milestones_timeline]
y_min, y_max = 2015.5, 2026.5

# Timeline axis line
ax.axhline(0, color='#333333', linewidth=3, zorder=2)

# Posizioni alternanti sopra/sotto
for i, (year, text, color) in enumerate(milestones_timeline):
    above = (i % 2 == 0)
    y_offset = 0.55 if above else -0.55
    y_text   = 0.85 if above else -0.85

    # Linea verticale
    ax.plot([year, year], [0, y_offset * 0.9], color=color, linewidth=2.5, zorder=3)
    
    # Cerchio sull'asse
    ax.scatter(year, 0, s=200, color=color, zorder=5, edgecolors='white', linewidth=2)
    
    # Box con testo
    ax.text(year, y_text, text,
            ha='center', va='center' if not above else 'bottom',
            fontsize=9, fontweight='bold' if year in [2020, 2024, 2021] else 'normal',
            color='white',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=color,
                      edgecolor='white', alpha=0.95),
            zorder=6, multialignment='center')
    
    # Etichetta year
    ax.text(year, 0.08 if above else -0.08, str(year),
            ha='center', va='bottom' if above else 'top',
            fontsize=10, fontweight='bold', color=color, zorder=6)

ax.set_xlim(y_min, y_max)
ax.set_ylim(-1.4, 1.4)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title('Timeline of Bioinformatics Milestones (2016–2026)',
             fontsize=17, fontweight='bold', pad=20)
fig.text(0.99, 0.01,
         'Sources: Nature, PubMed, AlphaFold DB, FDA, NHGRI, StartUs Insights, CSET Georgetown',
         ha='right', va='bottom', fontsize=8, color='gray', style='italic')
plt.tight_layout()
fig.savefig(f'{CHARTS_DIR}/chart8_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart 8 saved")


## 6. Analysis of Main Trends

---

### 1. The Era of AI in Bioinformatics

Publications on machine learning applied to bioinformatics have grown by **~291% in pharmacology** and by **~167% in genetics** between 2015 and 2021 (fonte: [CSET Georgetown](https://cset.georgetown.edu/publication/ai-and-the-future-of-life-sciences/)). Deep learning followed an even more vertical trajectory: from a few hundred papers in 2016 to tens of thousands in 2025.

Investment in AI for the biotech sector reached **$5,6 miliardi nel 2024**, con modelli fondazionali (AlphaFold, ESM2, scGPT, Evo) becoming de facto standards in computational laboratories. In 2026, agentic AI systems autonomously drive entire drug discovery pipelines.

---

### 2. Single-Cell and Spatial Omics

Le pubblicazioni sul sequenziamento single-cell hanno registrato un **181-fold increase from 2013 to 2023** ([IntechOpen](https://www.intechopen.com/books/1000107)). Spatial transcriptomics was named **Method of the Year 2020** by *Nature Methods* ([Nature](https://www.nature.com/articles/s41592-020-01042-x)), marking the beginning of a new era in the study of tissue biology.

In 2025, the **foundation models for single-cell** (scGPT, scFoundation, Geneformer) became the default approach for analyzing multi-million cell atlases. In 2026, spatial biology reaches the scalability needed for routine clinical applications.

---

### 3. Multi-Omics and Precision Medicine

The integration of genomics, transcriptomics, proteomics, and metabolomics (multi-omics) is the most active frontier. The **FDA approved 48 precision medicine indications in 2024**. Proteomics grows at a CAGR of **14,43%** ([Mordor Intelligence](https://www.mordorintelligence.com/industry-reports/proteomics-market)).

Next-generation sequencing (NGS) is now integrated into routine oncology pathways in many European countries, with multi-omic analyses driving personalized therapeutic decisions.

---

### 4. Genomics and Sequencing: The Collapse in Costs

The cost of sequencing a human genome plummeted from **$14 milioni nel 2006 a circa $200 nel 2024** ([NHGRI](https://www.genome.gov/about-genomics/fact-sheets/DNA-Sequencing-Costs-Data)) – a 70,000-fold reduction in 18 years, well below Moore's curve.

The market for **long-read sequencing** (PacBio HiFi, Oxford Nanopore R10) grows at **22,74% CAGR**, with accuracies above 99.9% now achievable. This enables the characterization of complex structural variants, epigenetic modifications, and complete transcriptomes impossible with short-read sequencing.

---

### 5. CRISPR and Gene Editing

Dalla prima approvazione terapeutica (Casgevy/CTX001 per anemia falciforme nel 2023), CRISPR ha attraversato una fase di matureszione rapida. Il **prime editing** expands the window of correctable variants, while in 2026 the first therapies of **base editing in vivo** for cardiovascular diseases enter advanced clinical trials.

Le pubblicazioni su "CRISPR AND bioinformatics" mostrano una crescita costante, trainata dall'esigenza di analisi off-target, progettazione di guide RNA e analisi degli effetti cellulari.

---

### 6. Cloud Computing and Data Infrastructure

The exponential growth of genomic data (estimated at **40 exabyte/year entro il 2025** according to some projections) makes cloud computing no longer optional but essential. Platforms like AWS HealthOmics, Google Cloud Life Sciences, and Azure Genomics have become standard infrastructures for enterprise bioinformatics.

Il **federated learning** emerges as a solution for the analysis of sensitive clinical genomic data without violating privacy, enabling consortium studies on a global scale.

---

### 7. Market and Employment

Il mercato globaland byla bioinformatica vale **$17,83 miliardi nel 2025** ([Nova One Advisor](https://www.novaoneadvisor.com/report/bioinformatics-market)) and projections indicate **$63,96 miliardi by 2035** (CAGR 13,63%). Bioinformatics roles consistently feature in the **top 5 most in-demand STEM professions**.

The most valued skill by the market is the combination of deep computational skills (Python, R, machine learning) with a solid biological understanding. Data scientists with a background in biology obtain salaries on average 15-25% higher than those without.


## 7. Key Takeaways for LinkedIn

---

**🚀 Numeri che fanno la differenza – Bioinformatica 2016→2026:**

- **$14.000.000 → $200**: the cost to sequence a human genome over 18 years. A 70,000x reduction that has democratized genomics. ([NHGRI](https://www.genome.gov/about-genomics/fact-sheets/DNA-Sequencing-Costs-Data))

- **+291%**: growth of AI/ML publications applied to pharmacology between 2015 and 2021. Artificial intelligence is not the future of bioinformatics – it is its present. ([CSET Georgetown](https://cset.georgetown.edu/publication/ai-and-the-future-of-life-sciences/))

- **200 milioni di strutture proteiche**: this is the number of structures predicted by AlphaFold DB at the time of its public release in 2022. By comparison, the global scientific community had resolved ~200,000 structures in 60 years of crystallography.

- **$5,6 miliardi**: investment in AI for biotech in 2024 alone. Bioinformatics is no longer an academic niche – it is a global strategic investment sector.

- **181x**: increase in single-cell sequencing publications from 2013 to 2023. We are mapping every cell in the human body – individually.

- **$17,83B → $63,96B**: the trajectory of the bioinformatics market from 2025 to 2035 (CAGR 13.63%). Few sectors offer this combination of scientific impact and economic growth. ([Nova One Advisor](https://www.novaoneadvisor.com/report/bioinformatics-market))

- **48 precision therapies FDA-approved in 2024**: personalized medicine is already a reality, and bioinformatics is its invisible but fundamental enabler.

- **Top 5 professioni STEM più richieste**: bioinformatics roles are among the most sought after by the job market, with a demand that structurally exceeds the supply of qualified talent.

---

**💡 Strategic conclusion:**

La bioinformatica si trova al crocevia di tre megatrend convergenti: l'esplosione dei dati biologici (genomica, single-cell, spatial), la matureszione dei modelli AI/ML per dati biologici complessi, e la pressione verso la medicina personalizzata. Chi padroneggia questa triplice competenza – biologia, informatica, intelligenza artificiale – è posizionato in uno dei punti di maggior leva per il prossimo decennio della scienza and byla medicina.


## 9. Distribution by Main Bioinformatics Journals

I giornali scientifici sono il cuore pulsantand byla ricerca bioinformatica. In questa sezione analizziamo:
- **Ranking** dei principali giornali per Impact Factor e h-index
- **Volume di pubblicazioni** e come è cambiato nel tempo
- **Distribuzione tematica** tra i giornali
- Query **PubMed** per contare le pubblicazioni per giornale nel tempo

### Sources dei dati
- [Google Scholar Metrics — Bioinformatics & Computational Biology](https://scholar.google.com/citations?view_op=top_venues&hl=en&vq=eng_bioinformatics)
- [SCImago Journal & Country Rank](https://www.scimagojr.com/)
- [Manusights — Journal Impact Factor Database](https://manusights.com/)
- [BioxBio — Journal Impact Factors](https://www.bioxbio.com/)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9a. MAIN BIOINFORMATICS JOURNALS DATA
# ════════════════════════════════════════════════════════════════════════════
# Sources: Google Scholar Metrics 2024, JCR 2024, SCImago 2023, Manusights 2026

import pandas as pd
import numpy as np

# ── Journal ranking (2024 JCR / Google Scholar h5-index data) ──────────
journals_ranking = pd.DataFrame({
    "Giornale": [
        "Nature Methods",
        "Nucleic Acids Research",
        "Genome Biology",
        "Briefings in Bioinformatics",
        "Bioinformatics (Oxford)",
        "Genome Research",
        "PLOS Computational Biology",
        "BMC Bioinformatics",
        "GigaScience",
        "Journal of Cheminformatics",
        "NAR Genomics and Bioinformatics",
        "IEEE/ACM Trans. Comp. Bio. & Bioinf.",
        "Genomics, Proteomics & Bioinformatics",
        "Computational Biology and Chemistry",
        "Database (Oxford)",
    ],
    "Impact Factor 2024": [32.1, 13.1, 9.4, 7.7, 5.4, 5.5, 3.8, 2.9, 5.0, 7.1, 4.0, 3.6, 11.5, 2.6, 3.4],
    "h5-index (Scholar)":  [107,  137,  92,  128, 137, 56,  95,  73,  56,  65,  45,  59,  49,  41,  39],
    "Quartile":            ["Q1","Q1","Q1","Q1","Q1","Q1","Q1","Q2","Q1","Q1","Q1","Q1","Q1","Q2","Q1"],
    "Focus Principale": [
        "Metodi sperimentali e computazionali",
        "Database, web server, acidi nucleici",
        "Genomica, metodi, risorse",
        "Review, tutorial bioinformatici",
        "Algoritmi, tool, software",
        "Genomica, sequenziamento",
        "Biologia computazionale",
        "Bioinformatica applicata",
        "Data science, big data",
        "Cheminformatica",
        "Genomica computazionale",
        "Metodi computazionali",
        "Genomica, proteomica",
        "Biologia computazionale",
        "Database biologici",
    ],
    "Editore": [
        "Springer Nature", "Oxford Univ. Press", "Springer Nature/BioMed Central",
        "Oxford Univ. Press", "Oxford Univ. Press", "Cold Spring Harbor",
        "PLOS", "Springer Nature/BioMed Central", "Oxford Univ. Press/GigaScience Press",
        "Springer Nature/BioMed Central", "Oxford Univ. Press",
        "IEEE/ACM", "Elsevier/Beijing Genomics Inst.", "Elsevier", "Oxford Univ. Press"
    ],
    "Citazioni Totali (approx.)": [
        "N/A", "290,054", "N/A", "N/A", "179,068",
        "N/A", "42,510", "N/A", "N/A", "N/A",
        "N/A", "N/A", "N/A", "N/A", "N/A"
    ]
})

print("=" * 80)
print("RANKING OF MAIN BIOINFORMATICS JOURNALS")
print("=" * 80)
display(journals_ranking[["Giornale", "Impact Factor 2024", "h5-index (Scholar)", "Quartile", "Focus Principale"]])


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9b. PUBMED QUERY — PUBLICATIONS BY JOURNAL OVER TIME
# ════════════════════════════════════════════════════════════════════════════
# Usiamo PubMed E-utilities per contare le pubblicazioni per giornale per year

import os, time, requests
import xml.etree.ElementTree as ET

JOURNAL_CACHE = "/home/user/workspace/pubmed_journals_cache.csv"

# Mapping: journal name → termine PubMed (journal abbreviation)
journal_pubmed_terms = {
    "Bioinformatics":             '"Bioinformatics"[Journal]',
    "Nucleic Acids Res.":         '"Nucleic Acids Research"[Journal]',
    "Briefings in Bioinf.":       '"Briefings in bioinformatics"[Journal]',
    "PLOS Comput. Biol.":         '"PLoS computational biology"[Journal]',
    "BMC Bioinformatics":         '"BMC bioinformatics"[Journal]',
    "Genome Biology":             '"Genome biology"[Journal]',
    "Genome Research":            '"Genome research"[Journal]',
    "Nature Methods":             '"Nature methods"[Journal]',
    "GigaScience":                '"GigaScience"[Journal]',
    "J. Cheminformatics":         '"Journal of cheminformatics"[Journal]',
    "Frontiers in Bioinf.":       '"Frontiers in bioinformatics"[Journal]',
    "NAR Genom. Bioinf.":         '"NAR genomics and bioinformatics"[Journal]',
}

years = list(range(2016, 2026))

def get_pubmed_count(term, year):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "rettype": "count",
        "mindate": f"{year}/01/01",
        "maxdate": f"{year}/12/31",
        "datetype": "pdat"
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        root = ET.fromstring(resp.content)
        count = int(root.find("Count").text)
        time.sleep(0.35)
        return count
    except Exception as e:
        print(f"  ⚠ Error for {term} ({year}): {e}")
        return None

if os.path.exists(JOURNAL_CACHE):
    print(f"✅ Cache found: {JOURNAL_CACHE}")
    journal_df = pd.read_csv(JOURNAL_CACHE, index_col=0)
else:
    print("🔄 Interrogazione PubMed per giornali (potrebbe richiedere ~2 minutes)...")
    rows = {}
    for jname, jterm in journal_pubmed_terms.items():
        print(f"  → {jname}...", end=" ")
        counts = []
        for y in years:
            c = get_pubmed_count(jterm, y)
            counts.append(c)
        rows[jname] = counts
        print(counts)
    journal_df = pd.DataFrame(rows, index=years)
    journal_df.index.name = "Year"
    journal_df.to_csv(JOURNAL_CACHE)
    print(f"💾 Cache saved: {JOURNAL_CACHE}")

print("\n📊 Pubblicazioni per giornale per year:")
display(journal_df)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# QUERY PUBMED — JOURNAL PUBLICATIONS OVER TIME
# ════════════════════════════════════════════════════════════════════════════

import os, time, requests
import xml.etree.ElementTree as ET

# ── Cache path: same directory as pubmed_cache.csv ───────────────────────────
JOURNAL_CACHE = os.path.join(os.path.dirname(CACHE_CSV), "pubmed_journals_cache.csv")

# ── Journal → PubMed search term mapping ─────────────────────────────────────
journal_pubmed_terms = {
    "Bioinformatics":          '"Bioinformatics"[Journal]',
    "Nucleic Acids Res.":      '"Nucleic Acids Research"[Journal]',
    "Briefings in Bioinf.":    '"Briefings in bioinformatics"[Journal]',
    "PLOS Comput. Biol.":      '"PLoS computational biology"[Journal]',
    "BMC Bioinformatics":      '"BMC bioinformatics"[Journal]',
    "Genome Biology":          '"Genome biology"[Journal]',
    "Genome Research":         '"Genome research"[Journal]',
    "Nature Methods":          '"Nature methods"[Journal]',
    "GigaScience":             '"GigaScience"[Journal]',
    "J. Cheminformatics":      '"Journal of cheminformatics"[Journal]',
    "Frontiers in Bioinf.":    '"Frontiers in bioinformatics"[Journal]',
    "NAR Genom. Bioinf.":      '"NAR genomics and bioinformatics"[Journal]',
}

years = list(range(2016, 2026))

def get_pubmed_count(term, year):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "rettype": "count",
        "mindate": f"{year}/01/01",
        "maxdate": f"{year}/12/31",
        "datetype": "pdat"
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        root = ET.fromstring(resp.content)
        count = int(root.find("Count").text)
        time.sleep(0.35)
        return count
    except Exception as e:
        print(f"  ⚠ Error for {term} ({year}): {e}")
        return None

# ── Load cache or query PubMed ────────────────────────────────────────────────
if os.path.exists(JOURNAL_CACHE) and os.path.getsize(JOURNAL_CACHE) > 0:
    print(f"✅ Cache found: {JOURNAL_CACHE}")
    journal_df = pd.read_csv(JOURNAL_CACHE, index_col=0)
else:
    print("🔄 Querying PubMed for journals (this may take ~2 minutes)...")
    rows = {}
    for jname, jterm in journal_pubmed_terms.items():
        print(f"  → {jname}...", end=" ")
        counts = []
        for y in years:
            c = get_pubmed_count(jterm, y)
            counts.append(c)
        rows[jname] = counts
        print(counts)
    journal_df = pd.DataFrame(rows, index=years)
    journal_df.index.name = "Year"
    journal_df.to_csv(JOURNAL_CACHE)
    print(f"💾 Cache saved: {JOURNAL_CACHE}")

print("\n📊 Publications per journal per year:")
display(journal_df)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 9a: JOURNAL RANKING BY IMPACT FACTOR
# ════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 8))

# Sort by IF
jr_sorted = journals_ranking.sort_values("Impact Factor 2024", ascending=True)

colors_if = []
for val in jr_sorted["Impact Factor 2024"]:
    if val >= 10:
        colors_if.append("#1a5276")    # top tier — dark blue
    elif val >= 5:
        colors_if.append("#2980b9")    # strong — medium blue
    else:
        colors_if.append("#85c1e9")    # good — light blue

bars = ax.barh(
    jr_sorted["Giornale"],
    jr_sorted["Impact Factor 2024"],
    color=colors_if,
    edgecolor="white",
    linewidth=0.5,
    height=0.7,
)

# Labels on bars
for bar, val in zip(bars, jr_sorted["Impact Factor 2024"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}", va="center", fontsize=11, fontweight="bold", color="#2c3e50")

ax.set_xlabel("Impact Factor (JCR 2024)", fontsize=13, fontweight="bold")
ax.set_title("Ranking of Main Bioinformatics Journals — Impact Factor 2024",
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlim(0, 36)
ax.grid(axis="x", alpha=0.3)
ax.tick_params(axis="y", labelsize=11)

# Tier legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#1a5276", label="Tier 1 (IF ≥ 10)"),
    Patch(facecolor="#2980b9", label="Tier 2 (IF 5–10)"),
    Patch(facecolor="#85c1e9", label="Tier 3 (IF < 5)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=11, framealpha=0.9)

plt.tight_layout()
fig.savefig(CHARTS_DIR + "/chart9a_journal_impact_factor.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Salvato: charts/chart9a_journal_impact_factor.png")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 9b: PUBLICATION TREND BY JOURNAL (2016–2025)
# ════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(16, 9))

# Distinguishable professional palette
journal_palette = [
    "#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12",
    "#1abc9c", "#e67e22", "#34495e", "#16a085", "#8e44ad",
    "#c0392b", "#2c3e50"
]

# Select top journals by average volume
mean_vols = journal_df.mean().sort_values(ascending=False)

for i, jname in enumerate(mean_vols.index):
    vals = journal_df[jname].values
    lw = 2.5 if i < 5 else 1.5
    alpha = 1.0 if i < 5 else 0.6
    marker = "o" if i < 5 else "s"
    ax.plot(years, vals, marker=marker, linewidth=lw, alpha=alpha,
            color=journal_palette[i % len(journal_palette)],
            label=jname, markersize=5)

ax.set_xlabel("Year", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Publications", fontsize=13, fontweight="bold")
ax.set_title("Evoluzionand byle Pubblicazioni nei Principali Giornali di Bioinformatica (2016–2025)",
             fontsize=14, fontweight="bold", pad=15)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10, framealpha=0.9)
ax.set_xticks(years)
ax.grid(alpha=0.3)
ax.tick_params(labelsize=11)

plt.tight_layout()
fig.savefig(CHARTS_DIR + "/chart9b_journal_publication_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Salvato: charts/chart9b_journal_publication_trends.png")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 9c: EDITORIAL MARKET SHARES — 2016 vs 2025
# ════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

def make_pie(ax, year_data, title):
    # Ordina e raggruppa i piccoli in "Others"
    sorted_data = year_data.sort_values(ascending=False)
    threshold = sorted_data.sum() * 0.03  # sotto il 3% → "Others"
    main = sorted_data[sorted_data >= threshold]
    others = sorted_data[sorted_data < threshold].sum()
    if others > 0:
        main["Others"] = others

    colors_pie = [
        "#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12",
        "#1abc9c", "#e67e22", "#34495e", "#16a085", "#8e44ad",
        "#c0392b", "#95a5a6", "#d35400"
    ]

    wedges, texts, autotexts = ax.pie(
        main.values, labels=main.index, autopct=lambda p: f"{p:.1f}%" if p > 4 else "",
        colors=colors_pie[:len(main)],
        textprops={"fontsize": 9},
        pctdistance=0.8,
        startangle=140,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5}
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_fontweight("bold")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=10)

# Dati per year
if 2016 in journal_df.index and 2025 in journal_df.index:
    data_2016 = journal_df.loc[2016]
    data_2025 = journal_df.loc[2025]
elif journal_df.index.min() == 2016:
    data_2016 = journal_df.iloc[0]
    data_2025 = journal_df.iloc[-1]
else:
    data_2016 = journal_df.iloc[0]
    data_2025 = journal_df.iloc[-1]

make_pie(axes[0], data_2016, f"Distribuzione Pubblicazioni ({journal_df.index.min()})")
make_pie(axes[1], data_2025, f"Distribuzione Pubblicazioni ({journal_df.index.max()})")

plt.suptitle("How Editorial Distribution in Bioinformatics has Changed",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(CHARTS_DIR + "/chart9c_journal_share_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Salvato: charts/chart9c_journal_share_comparison.png")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 9d: HEATMAP — PUBLICATION INTENSITY BY JOURNAL AND YEAR
# ════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(16, 8))

# Normalize by row (min-max) to make journals of different sizes comparable
journal_norm = journal_df.T.copy().astype(float)
for idx in journal_norm.index:
    row = journal_norm.loc[idx]
    rmin, rmax = row.min(), row.max()
    if rmax > rmin:
        journal_norm.loc[idx] = (row - rmin) / (rmax - rmin)
    else:
        journal_norm.loc[idx] = 0.5

# Sort by descending average volume
order = journal_df.mean().sort_values(ascending=False).index
journal_norm = journal_norm.loc[order]

sns.heatmap(
    journal_norm, yeart=journal_df.T.loc[order], fmt=".0f",
    cmap="YlOrRd", linewidths=0.5, linecolor="white",
    ax=ax, cbar_kws={"label": "Relative intensity (normalized by journal)", "shrink": 0.8},
    yeart_kws={"fontsize": 9}
)

ax.set_title("Publication Intensity by Journal (2016–2025)\n"
             "Valori: conteggio assoluto | Colore: intensità relativa per giornale",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Year", fontsize=12, fontweight="bold")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=11)
ax.tick_params(axis="x", labelsize=11)

plt.tight_layout()
fig.savefig(CHARTS_DIR + "/chart9d_journal_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Salvato: charts/chart9d_journal_heatmap.png")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CHART 9e: % GROWTH OF JOURNALS (2016 → 2025) + NUOVE RIVISTE
# ════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [2, 1.2]})

# ── Left panel: crescita % ──
first_year = journal_df.index.min()
last_year = journal_df.index.max()

growth = {}
for jname in journal_df.columns:
    v_start = journal_df[jname].iloc[0]
    v_end = journal_df[jname].iloc[-1]
    if v_start and v_start > 0:
        growth[jname] = ((v_end - v_start) / v_start) * 100
    else:
        # Giornale nato dopo il 2016 — segna come "New"
        growth[jname] = None

# Separate existing journals from new ones
existing = {k: v for k, v in growth.items() if v is not None}
new_journals = {k: journal_df[k].iloc[-1] for k, v in growth.items() if v is None}

gs = pd.Series(existing).sort_values(ascending=True)

colors_growth = ["#27ae60" if v >= 0 else "#e74c3c" for v in gs.values]

bars = axes[0].barh(gs.index, gs.values, color=colors_growth, edgecolor="white", height=0.6)
for bar, val in zip(bars, gs.values):
    xpos = bar.get_width() + 2 if val >= 0 else bar.get_width() - 2
    ha = "left" if val >= 0 else "right"
    axes[0].text(xpos, bar.get_y() + bar.get_height()/2,
                 f"{val:+.0f}%", va="center", ha=ha, fontsize=11, fontweight="bold",
                 color="#2c3e50")

axes[0].set_xlabel(f"% Variation ({first_year} → {last_year})", fontsize=12, fontweight="bold")
axes[0].set_title(f"Growth dei Giornali Bioinformatici\n({first_year} → {last_year})",
                   fontsize=14, fontweight="bold")
axes[0].axvline(x=0, color="black", linewidth=0.8)
axes[0].grid(axis="x", alpha=0.3)
axes[0].tick_params(axis="y", labelsize=11)

# ── Right panel: nuove riviste nate nel periodo ──
new_journals_info = {
    "Frontiers in Bioinf.\n(fondata 2021)": "2021",
    "NAR Genom. Bioinf.\n(fondata 2019)": "2019",
    "iScience (Bioinf.)\n(fondata 2018)": "2018",
    "Bioinformatics Advances\n(fondata 2021)": "2021",
}

y_positions = range(len(new_journals_info))
colors_new = ["#3498db", "#9b59b6", "#f39c12", "#1abc9c"]

axes[1].barh(list(new_journals_info.keys()), [1]*len(new_journals_info),
             color=colors_new[:len(new_journals_info)], edgecolor="white", height=0.5)

for i, (name, yr) in enumerate(new_journals_info.items()):
    axes[1].text(0.5, i, f"Year: {yr}", va="center", ha="center",
                 fontsize=12, fontweight="bold", color="white")

axes[1].set_xlim(0, 1)
axes[1].set_xticks([])
axes[1].set_title("New Bioinformatics Journals\n(2016–2026)", fontsize=14, fontweight="bold")
axes[1].tick_params(axis="y", labelsize=10)

plt.suptitle("Editorial Dynamics in Bioinformatics", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(CHARTS_DIR + "/chart9e_journal_growth.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Salvato: charts/chart9e_journal_growth.png")


## 9f. Analysis of Editorial Distribution

### Key Observations

**Journal Hierarchy:**
- **Tier 1 (IF ≥ 10):** *Nature Methods* (IF 32.1) clearly dominates, followed by *Nucleic Acids Research* (IF 13.1), *Genome Biology* (IF 9.4), *Genomics Proteomics & Bioinformatics* (IF 11.5)
- **Tier 2 (IF 5–10):** *Briefings in Bioinformatics* (IF 7.7), *Bioinformatics* (IF 5.4), *Genome Research* (IF 5.5), *J. Cheminformatics* (IF 7.1)
- **Tier 3 (IF < 5):** *PLOS Comp. Bio.* (IF 3.8), *BMC Bioinformatics* (IF 2.9), *Database* (IF 3.4)

**Editorial Trends (2016–2025):**
- *Bioinformatics* (Oxford) remains the most cited journal worldwide in computational biology with **2.7 milioni di total citations** e un h-index di **564**
- *Nucleic Acids Research* has the highest annual citation volume (**290,054 total citations**), thanks to Database Issue papers
- Birth of new specialized journals: *Frontiers in Bioinformatics* (2021), *NAR Genomics and Bioinformatics* (2019), *Bioinformatics Advances* (2021)
- The **Open Access** model is growing: *Frontiers*, *PLOS*, *BMC*, *GigaScience* are all OA

**Concentrazionand by Campo:**
- I primi 5 giornali specializzati coprono la maggior partand byle pubblicazioni bioinformatiche "pure"
- However, a lot of bioinformatics is also published in generalist journals (*Nature*, *Science*, *Cell*) and genomics (*Nature Genetics*, *Cell Genomics*)
- Google Scholar ranking: *Bioinformatics* e *Nucleic Acids Research* share the first place with **h5-index di 137**

**Sources:**
- [Google Scholar Metrics — Bioinformatics & Computational Biology](https://scholar.google.com/citations?view_op=top_venues&hl=en&vq=eng_bioinformatics)
- [Manusights — Bioinformatics Impact Factor 2026](https://manusights.com/blog/bioinformatics-impact-factor)
- [Manusights — Nucleic Acids Research Impact Factor 2026](https://manusights.com/blog/nucleic-acids-research-impact-factor)
- [SCImago — Nature Methods](https://www.scimagojr.com/journalsearch.php?q=21100778827&tip=sid)
- [BioxBio — PLOS Computational Biology](https://www.bioxbio.com/journal/PLOS-COMPUT-BIOL)


## 8. Sources e Riferimenti

| # | Source | URL | Type |
|---|-------|-----|------|
| 1 | PubMed E-utilities API (NCBI) | https://www.ncbi.nlm.nih.gov/books/NBK25499/ | API |
| 2 | NHGRI – DNA Sequencing Costs | https://www.genome.gov/about-genomics/fact-sheets/DNA-Sequencing-Costs-Data | Official data |
| 3 | StartUs Insights – Bioinformatics Trends | https://www.startus-insights.com/innovators-guide/bioinformatics-trends/ | Report |
| 4 | CSET Georgetown – AI and Life Sciences | https://cset.georgetown.edu/publication/ai-and-the-future-of-life-sciences/ | Report accademico |
| 5 | ZAGENO – Bioinformatics Trends | https://zageno.com/l/bioinformatics-trends | Industry blog |
| 6 | Fios Genomics – Bioinformatics Trends | https://fiosgenomics.com/bioinformatics-trends/ | Industry blog |
| 7 | PharmiWeb / SNS Insider – Bioinformatics Market 2023 | https://www.pharmiweb.com/press-release/2024-07-16/bioinformatics-market-size-worth-usd-13614-million-in-2023 | Press release |
| 8 | Coherent Market Insights – Bioinformatics Market | https://www.coherentmarketinsights.com/market-insight/bioinformatics-market-2177 | Market report |
| 9 | BioSpace – Bioinformatics Market | https://www.biospace.com/bioinformatics-market | Market report |
| 10 | Nova One Advisor – Bioinformatics Market | https://www.novaoneadvisor.com/report/bioinformatics-market | Market report |
| 11 | Nature Methods – Method of the Year 2020 | https://www.nature.com/articles/s41592-020-01042-x | Journal |
| 12 | IntechOpen – Single-Cell Bioinformatics | https://www.intechopen.com/books/1000107 | Review book |
| 13 | Mordor Intelligence – Proteomics Market | https://www.mordorintelligence.com/industry-reports/proteomics-market | Market report |
| 14 | Precedence Research – Bioinformatics Market | https://www.precedenceresearch.com/bioinformatics-market | Market report |
| 15 | AlphaFold DB | https://alphafold.ebi.ac.uk/ | Database |
| 16 | LinkedIn – Andy Hickl (AI in Bioinformatics) | https://www.linkedin.com/pulse/future-ai-bioinformatics-2026-andy-hickl/ | LinkedIn article |

---

*Notebook created: Aprile 2026 | Author: Bioinformatics analysis for LinkedIn | Python 3.10+*


### Sources Aggiuntive — Analisi Giornali
- **Google Scholar Metrics** — Bioinformatics & Computational Biology: https://scholar.google.com/citations?view_op=top_venues&hl=en&vq=eng_bioinformatics
- **Manusights** — Journal Impact Factor Database: https://manusights.com/
- **SCImago Journal & Country Rank**: https://www.scimagojr.com/
- **BioxBio** — Journal Impact Factors: https://www.bioxbio.com/
- **Manusights** — Bioinformatics IF 2026: https://manusights.com/blog/bioinformatics-impact-factor
- **Manusights** — NAR IF 2026: https://manusights.com/blog/nucleic-acids-research-impact-factor
- **Manusights** — Top Genomics & Methods Journals: https://manusights.com/journals/field/genomics-methods


In [ ]:
# ── Final Summary ──────────────────────────────────────────────────────────
import glob as glob_module

charts = sorted(glob_module.glob(f'{CHARTS_DIR}/*.png'))
print(f"\n✅ Notebook completed — {len(charts)} charts saved in {CHARTS_DIR}/")
for c in charts:
    print(f"   {os.path.basename(c)}")
print("\n📊 PubMed DataFrame (preview):")
display(df_pubmed.style.background_gradient(cmap='Blues', axis=1))
